# Git-Clone Workflow Template — Kaggle

**This is a template, not a tested notebook** — it needs a real pushed GitHub repository to actually run, which doesn't exist until you push this project there. Replace `<YOUR-GITHUB-URL>` below once you have.

**Why this is worth switching to**: the three per-method notebooks in this folder recreate every file via `%%writefile` cells. That works, but it means updating any file requires editing that specific cell AND remembering to re-run it before your next training command -- easy to forget, and exactly the failure mode that came up repeatedly earlier in this project (stale files from an un-rerun cell). With `git clone` + `git pull`, the whole project updates with one command, and there's no way to accidentally train against a stale version of one file while others are current.


## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


## 2. Clone the repo

Requires **Settings > Internet > On**. Replace the URL with your actual GitHub repo once pushed.


In [ ]:
REPO_URL = 'https://github.com/<YOUR-USERNAME>/<YOUR-REPO-NAME>.git'  # <- replace this

import os
if os.path.isdir('/kaggle/working/repo'):
    print('Repo already cloned -- pulling latest changes instead.')
    %cd /kaggle/working/repo
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL} repo
    %cd repo


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 4. Get shared data

Real old photos: attach your uploaded Kaggle Dataset via **Add Data**, same as in the per-method notebooks. Clean photos (VOC2012) and damage masks: same commands as the README's "One-time shared data preparation" section -- run once, reused by all three methods.


In [ ]:
REAL_PHOTO_DIR = '/kaggle/input/real-old-photos-vae1'  # <- adjust to your dataset slug

import torchvision.datasets as tvds
VOC_ROOT = '/kaggle/working/repo/data/voc2012'
VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')
if not (os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0):
    os.makedirs(VOC_ROOT, exist_ok=True)
    _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)
print('VOC2012 ready at', VOC_JPEG_DIR)

MASKS_DIR = '/kaggle/working/repo/data/generated_masks'
os.makedirs(MASKS_DIR, exist_ok=True)
if not os.path.isdir('FilmDamageSimulator'):
    !git clone --depth 1 https://github.com/daniela997/FilmDamageSimulator.git
!cp data_tools/damage_generation/generate_synthetic_only.py FilmDamageSimulator/damage_generator/
%cd FilmDamageSimulator/damage_generator
!python generate_synthetic_only.py --types scratches,smut --height 256 --width 256 --n 3000
%cd /kaggle/working/repo
!cp FilmDamageSimulator/generated/*.png {MASKS_DIR}/
print('Masks ready at', MASKS_DIR)


## 5. Run any method directly

No file recreation needed -- the repo on disk IS the source of truth. Same `python -m` commands as the README.


In [ ]:
# Method 1 (three stages) -- see README for the full sequence
# !python -m method1_gan_vae.train_vae_domain_a \
#     --data-root "$REAL_PHOTO_DIR" --epochs 50 --batch-size 16 --image-size 256 \
#     --amp --out-dir ./runs/vae_domain_a --device cuda

# Method 2
# !python -m method2_diffbir.train_stage1_restoration \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 50 --batch-size 16 --image-size 256 \
#     --amp --out-dir ./runs/stage1_restoration --device cuda

# Method 3
# !python -m method3_transformer.train_transformer_regression \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 50 --batch-size 4 --image-size 256 \
#     --embed-dim 60 --depths "4,4,4,4" --num-heads 6 --window-size 8 \
#     --use-checkpoint --amp --out-dir ./runs/transformer_regression --device cuda


## Updating the project mid-session

If you push a fix to GitHub while a Kaggle session is running, just re-run this to pull it in -- no cell-by-cell file recreation needed:


In [ ]:
%cd /kaggle/working/repo
!git pull
